# Urban Ecosystem Dynamics Predictor

## Project Overview
This project builds a predictive model to simulate short-term ecosystem dynamics in urban areas, focusing on:
- Tree canopy cover
- Air quality index
- Soil moisture levels
- Micro-temperature variations

The model enables **what-if scenarios** like:
- "What if we plant more trees here?"
- "What if the green cover decreases?"

**Dataset**: Simulated data based on characteristics of urban parks in India

**Author**: Mohammed Tofiq
**Date**: July 2025

## 1. Import Required Libraries
We'll use these libraries for data manipulation, modeling, and visualization

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Machine learning libraries
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# For creating interactive widgets
from ipywidgets import interact, FloatSlider, IntSlider
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

## 2. Generate Realistic Urban Park Dataset
Since real-time ecosystem data is limited, we'll create a realistic dataset based on typical urban park characteristics in India

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data for 365 days (1 year) of urban park ecosystem data
n_samples = 365

# Create base features that interact with each other realistically
def generate_ecosystem_data(n_samples):
    """
    Generate realistic urban ecosystem data with interdependent variables
    Based on typical conditions in Indian urban parks
    """
    
    # Base seasonal pattern (0-365 days)
    days = np.arange(n_samples)
    seasonal_pattern = np.sin(2 * np.pi * days / 365)
    
    # Tree canopy cover (30-85%) - varies seasonally
    canopy_cover = 60 + 15 * seasonal_pattern + np.random.normal(0, 5, n_samples)
    canopy_cover = np.clip(canopy_cover, 30, 85)
    
    # Air Quality Index (50-200) - inversely related to canopy cover
    # Lower values = better air quality
    air_quality = 150 - 0.8 * canopy_cover + np.random.normal(0, 15, n_samples)
    air_quality = np.clip(air_quality, 50, 200)
    
    # Soil moisture (20-80%) - related to canopy cover and seasonal patterns
    soil_moisture = 40 + 0.3 * canopy_cover + 10 * seasonal_pattern + np.random.normal(0, 8, n_samples)
    soil_moisture = np.clip(soil_moisture, 20, 80)
    
    # Micro-temperature (18-42°C) - inversely related to canopy cover
    base_temp = 30 + 8 * (-seasonal_pattern)  # Hotter in winter months in some regions
    micro_temperature = base_temp - 0.2 * canopy_cover + np.random.normal(0, 2, n_samples)
    micro_temperature = np.clip(micro_temperature, 18, 42)
    
    # Target variable: Future canopy cover (next month prediction)
    # Influenced by current conditions and seasonal trends
    future_canopy = (0.7 * canopy_cover + 
                    0.1 * (100 - air_quality) +  # Better air quality helps growth
                    0.15 * soil_moisture +        # More moisture helps growth
                    0.05 * (40 - micro_temperature) +  # Moderate temps help growth
                    5 * seasonal_pattern +        # Seasonal growth pattern
                    np.random.normal(0, 3, n_samples))
    
    future_canopy = np.clip(future_canopy, 25, 90)
    
    return pd.DataFrame({
        'day': days,
        'canopy_cover': canopy_cover,
        'air_quality_index': air_quality,
        'soil_moisture': soil_moisture,
        'micro_temperature': micro_temperature,
        'future_canopy_cover': future_canopy
    })

# Generate the dataset
data = generate_ecosystem_data(n_samples)

print(f"✅ Dataset created with {len(data)} samples")
print(f"📊 Dataset shape: {data.shape}")
print("\n📋 First 5 rows:")
print(data.head())

## 3. Data Exploration and Visualization
Let's understand our dataset better through visualizations

In [ ]:
# Basic statistical summary
print("📈 Statistical Summary:")
print(data.describe().round(2))

# Check for missing values
print(f"\n🔍 Missing values: {data.isnull().sum().sum()}")

# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Urban Ecosystem Dataset Overview', fontsize=16, fontweight='bold')

# Plot 1: Time series of all variables
axes[0, 0].plot(data['day'], data['canopy_cover'], label='Canopy Cover', alpha=0.8)
axes[0, 0].plot(data['day'], data['future_canopy_cover'], label='Future Canopy', alpha=0.8)
axes[0, 0].set_title('Canopy Cover Over Time')
axes[0, 0].set_xlabel('Day of Year')
axes[0, 0].set_ylabel('Coverage (%)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Air Quality Index
axes[0, 1].plot(data['day'], data['air_quality_index'], color='orange', alpha=0.8)
axes[0, 1].set_title('Air Quality Index Over Time')
axes[0, 1].set_xlabel('Day of Year')
axes[0, 1].set_ylabel('AQI (lower is better)')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Soil Moisture
axes[0, 2].plot(data['day'], data['soil_moisture'], color='brown', alpha=0.8)
axes[0, 2].set_title('Soil Moisture Over Time')
axes[0, 2].set_xlabel('Day of Year')
axes[0, 2].set_ylabel('Moisture (%)')
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Micro-temperature
axes[1, 0].plot(data['day'], data['micro_temperature'], color='red', alpha=0.8)
axes[1, 0].set_title('Micro-temperature Over Time')
axes[1, 0].set_xlabel('Day of Year')
axes[1, 0].set_ylabel('Temperature (°C)')
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Correlation heatmap
correlation_matrix = data.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='RdYlBu_r', center=0, 
            ax=axes[1, 1], fmt='.2f')
axes[1, 1].set_title('Feature Correlations')

# Plot 6: Distribution of target variable
axes[1, 2].hist(data['future_canopy_cover'], bins=30, alpha=0.7, color='green')
axes[1, 2].set_title('Distribution of Future Canopy Cover')
axes[1, 2].set_xlabel('Future Canopy Cover (%)')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔗 Key Correlations with Future Canopy Cover:")
correlations = correlation_matrix['future_canopy_cover'].sort_values(ascending=False)
for feature, corr in correlations.items():
    if feature != 'future_canopy_cover':
        print(f"  • {feature}: {corr:.3f}")

## 4. Data Preprocessing
Prepare the data for machine learning

In [ ]:
# Define features (input variables) and target (what we want to predict)
feature_columns = ['canopy_cover', 'air_quality_index', 'soil_moisture', 'micro_temperature']
target_column = 'future_canopy_cover'

# Separate features and target
X = data[feature_columns].copy()
y = data[target_column].copy()

print(f"📊 Features shape: {X.shape}")
print(f"🎯 Target shape: {y.shape}")

# Split data into training and testing sets
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(f"\n🏋️ Training set: {X_train.shape[0]} samples")
print(f"🧪 Testing set: {X_test.shape[0]} samples")

# Optional: Scale features for better model performance
# We'll keep the original scale for interpretability, but here's how you would scale:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Data preprocessing completed!")
print("\n📋 Feature summary:")
for i, col in enumerate(feature_columns):
    print(f"  • {col}: {X_train[col].min():.1f} - {X_train[col].max():.1f}")

## 5. Model Training and Evaluation
Train a Random Forest model to predict future canopy cover

In [ ]:
# Initialize Random Forest Regressor
# Random Forest is chosen because it:
# 1. Handles non-linear relationships well
# 2. Provides feature importance
# 3. Is robust to outliers
# 4. Doesn't require feature scaling

model = RandomForestRegressor(
    n_estimators=100,        # Number of trees in the forest
    max_depth=10,            # Maximum depth of trees
    min_samples_split=5,     # Minimum samples required to split a node
    min_samples_leaf=2,      # Minimum samples required at a leaf node
    random_state=42,         # For reproducibility
    n_jobs=-1               # Use all available processors
)

print("🌳 Training Random Forest model...")

# Train the model
model.fit(X_train, y_train)

print("✅ Model training completed!")

# Make predictions on both training and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate performance metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("\n📊 Model Performance:")
print(f"  • Training R² Score: {train_r2:.4f}")
print(f"  • Testing R² Score: {test_r2:.4f}")
print(f"  • Training RMSE: {train_rmse:.3f}%")
print(f"  • Testing RMSE: {test_rmse:.3f}%")

# Check for overfitting
if train_r2 - test_r2 > 0.1:
    print("⚠️  Model might be overfitting (large gap between train and test scores)")
else:
    print("✅ Model shows good generalization!")

## 6. Feature Importance Analysis
Understand which factors most influence future canopy cover

In [ ]:
# Get feature importances from the trained model
feature_importances = model.feature_importances_
feature_names = feature_columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

print("🎯 Feature Importance Ranking:")
for i, row in importance_df.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f} ({row['importance']*100:.1f}%)")

# Visualize feature importances
plt.figure(figsize=(12, 6))
bars = plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Feature Importance')
plt.title('Feature Importance in Predicting Future Canopy Cover', fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{width:.3f}', ha='left', va='center')

plt.tight_layout()
plt.show()

# Model prediction visualization
plt.figure(figsize=(12, 5))

# Plot 1: Predicted vs Actual for test set
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_test_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Future Canopy Cover (%)')
plt.ylabel('Predicted Future Canopy Cover (%)')
plt.title(f'Prediction Accuracy (R² = {test_r2:.3f})')
plt.grid(True, alpha=0.3)

# Plot 2: Residuals (prediction errors)
plt.subplot(1, 2, 2)
residuals = y_test - y_test_pred
plt.scatter(y_test_pred, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Future Canopy Cover (%)')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. What-If Scenario Simulation Functions
Create functions to simulate different scenarios

In [ ]:
def simulate_scenario(base_data, canopy_change=0, air_quality_change=0, 
                     soil_moisture_change=0, temperature_change=0):
    """
    Simulate what-if scenarios by adjusting environmental parameters
    
    Parameters:
    -----------
    base_data : DataFrame or dict
        Base environmental conditions
    canopy_change : float
        Change in canopy cover (percentage points)
    air_quality_change : float
        Change in air quality index (positive = worse air quality)
    soil_moisture_change : float
        Change in soil moisture (percentage points)
    temperature_change : float
        Change in micro-temperature (degrees Celsius)
    
    Returns:
    --------
    dict : Simulation results with predictions
    """
    
    # If base_data is a DataFrame, use the mean values
    if isinstance(base_data, pd.DataFrame):
        base_conditions = {
            'canopy_cover': base_data['canopy_cover'].mean(),
            'air_quality_index': base_data['air_quality_index'].mean(),
            'soil_moisture': base_data['soil_moisture'].mean(),
            'micro_temperature': base_data['micro_temperature'].mean()
        }
    else:
        base_conditions = base_data.copy()
    
    # Apply changes to create new scenario
    new_conditions = {
        'canopy_cover': base_conditions['canopy_cover'] + canopy_change,
        'air_quality_index': base_conditions['air_quality_index'] + air_quality_change,
        'soil_moisture': base_conditions['soil_moisture'] + soil_moisture_change,
        'micro_temperature': base_conditions['micro_temperature'] + temperature_change
    }
    
    # Ensure values stay within realistic bounds
    new_conditions['canopy_cover'] = np.clip(new_conditions['canopy_cover'], 0, 100)
    new_conditions['air_quality_index'] = np.clip(new_conditions['air_quality_index'], 0, 500)
    new_conditions['soil_moisture'] = np.clip(new_conditions['soil_moisture'], 0, 100)
    new_conditions['micro_temperature'] = np.clip(new_conditions['micro_temperature'], 0, 50)
    
    # Prepare data for prediction
    scenario_data = pd.DataFrame([new_conditions])
    
    # Make prediction
    predicted_future_canopy = model.predict(scenario_data)[0]
    
    # Calculate baseline prediction for comparison
    baseline_data = pd.DataFrame([base_conditions])
    baseline_prediction = model.predict(baseline_data)[0]
    
    # Calculate the impact
    impact = predicted_future_canopy - baseline_prediction
    
    return {
        'baseline_conditions': base_conditions,
        'new_conditions': new_conditions,
        'baseline_prediction': baseline_prediction,
        'scenario_prediction': predicted_future_canopy,
        'impact': impact,
        'impact_percentage': (impact / baseline_prediction) * 100 if baseline_prediction != 0 else 0
    }

def print_scenario_results(results, scenario_name="Scenario"):
    """
    Print scenario results in a formatted way
    """
    print(f"\n🌳 {scenario_name} Results:")
    print("="*50)
    
    print("📊 Current Conditions:")
    for key, value in results['baseline_conditions'].items():
        print(f"  • {key.replace('_', ' ').title()}: {value:.1f}")
    
    print("\n🔄 Modified Conditions:")
    for key, value in results['new_conditions'].items():
        change = value - results['baseline_conditions'][key]
        change_symbol = "📈" if change > 0 else "📉" if change < 0 else "➡️"
        print(f"  • {key.replace('_', ' ').title()}: {value:.1f} ({change_symbol} {change:+.1f})")
    
    print("\n🎯 Predictions:")
    print(f"  • Baseline Future Canopy: {results['baseline_prediction']:.1f}%")
    print(f"  • Scenario Future Canopy: {results['scenario_prediction']:.1f}%")
    
    impact_symbol = "🟢" if results['impact'] > 0 else "🔴" if results['impact'] < 0 else "🟡"
    print(f"  • Impact: {impact_symbol} {results['impact']:+.1f}% ({results['impact_percentage']:+.1f}%)")

print("✅ Simulation functions created successfully!")

## 8. Example What-If Scenarios
Let's test different scenarios that urban planners might be interested in

In [ ]:
# Use the test data as our baseline for scenarios
baseline_data = X_test

# Scenario 1: Increase tree planting (more canopy cover)
print("🌱 SCENARIO 1: TREE PLANTING INITIATIVE")
tree_planting_result = simulate_scenario(
    baseline_data, 
    canopy_change=15,  # Increase canopy cover by 15%
    soil_moisture_change=5  # Planting might improve soil moisture slightly
)
print_scenario_results(tree_planting_result, "Tree Planting Initiative")

# Scenario 2: Urban development (decrease in green cover)
print("\n\n🏢 SCENARIO 2: URBAN DEVELOPMENT")
development_result = simulate_scenario(
    baseline_data,
    canopy_change=-20,  # Decrease canopy cover by 20%
    air_quality_change=25,  # Worse air quality due to development
    soil_moisture_change=-10,  # Less soil moisture due to concrete
    temperature_change=2  # Higher temperature due to urban heat island
)
print_scenario_results(development_result, "Urban Development")

# Scenario 3: Climate change impact (higher temperatures)
print("\n\n🌡️ SCENARIO 3: CLIMATE CHANGE IMPACT")
climate_change_result = simulate_scenario(
    baseline_data,
    temperature_change=3,  # 3°C temperature increase
    soil_moisture_change=-8,  # Drier soil due to heat
    air_quality_change=10  # Slightly worse air quality
)
print_scenario_results(climate_change_result, "Climate Change Impact")

# Scenario 4: Comprehensive green initiative
print("\n\n🌿 SCENARIO 4: COMPREHENSIVE GREEN INITIATIVE")
green_initiative_result = simulate_scenario(
    baseline_data,
    canopy_change=25,  # Major increase in tree cover
    air_quality_change=-30,  # Significant air quality improvement
    soil_moisture_change=12,  # Better soil management
    temperature_change=-1.5  # Cooling effect from more trees
)
print_scenario_results(green_initiative_result, "Comprehensive Green Initiative")

# Create a comparison visualization
scenarios = {
    'Current': 0,
    'Tree Planting': tree_planting_result['impact'],
    'Urban Development': development_result['impact'],
    'Climate Change': climate_change_result['impact'],
    'Green Initiative': green_initiative_result['impact']
}

plt.figure(figsize=(12, 6))
colors = ['gray', 'green', 'red', 'orange', 'darkgreen']
bars = plt.bar(scenarios.keys(), scenarios.values(), color=colors, alpha=0.7)

plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.title('Impact of Different Scenarios on Future Canopy Cover', fontsize=14, fontweight='bold')
plt.ylabel('Change in Future Canopy Cover (%)')
plt.xlabel('Scenario')
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + (0.1 if height >= 0 else -0.3),
             f'{height:.1f}%', ha='center', va='bottom' if height >= 0 else 'top',
             fontweight='bold')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Interactive What-If Scenario Explorer
Use sliders to explore different scenarios interactively

In [ ]:
# Create an interactive widget for exploring scenarios
def interactive_scenario_explorer(canopy_change, air_quality_change, 
                                 soil_moisture_change, temperature_change):
    """
    Interactive function for exploring what-if scenarios
    """
    
    # Run simulation with current slider values
    result = simulate_scenario(
        baseline_data,
        canopy_change=canopy_change,
        air_quality_change=air_quality_change,
        soil_moisture_change=soil_moisture_change,
        temperature_change=temperature_change
    )
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Current vs New Conditions
    conditions = ['Canopy Cover', 'Air Quality', 'Soil Moisture', 'Temperature']
    baseline_values = [result['baseline_conditions']['canopy_cover'],
                      result['baseline_conditions']['air_quality_index'],
                      result['baseline_conditions']['soil_moisture'],
                      result['baseline_conditions']['micro_temperature']]
    
    new_values = [result['new_conditions']['canopy_cover'],
                  result['new_conditions']['air_quality_index'],
                  result['new_conditions']['soil_moisture'],
                  result['new_conditions']['micro_temperature']]
    
    x = np.arange(len(conditions))
    width = 0.35
    
    ax1.bar(x - width/2, baseline_values, width, label='Current', alpha=0.7)
    ax1.bar(x + width/2, new_values, width, label='Modified', alpha=0.7)
    
    ax1.set_xlabel('Environmental Factors')
    ax1.set_ylabel('Values')
    ax1.set_title('Current vs Modified Conditions')
    ax1.set_xticks(x)
    ax1.set_xticklabels(conditions, rotation=45)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot 2: Prediction Impact
    predictions = ['Current Prediction', 'New Prediction']
    pred_values = [result['baseline_prediction'], result['scenario_prediction']]
    colors = ['lightblue', 'green' if result['impact'] > 0 else 'red']
    
    bars = ax2.bar(predictions, pred_values, color=colors, alpha=0.7)
    ax2.set_ylabel('Future Canopy Cover (%)')
    ax2.set_title(f'Prediction Impact: {result["impact"]:+.1f}% ({result["impact_percentage"]:+.1f}%)')
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, pred_values):
        ax2.text(bar.get_x() + bar.get_width()/2., value + 0.5,
                f'{value:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    impact_emoji = "🟢" if result['impact'] > 0 else "🔴" if result['impact'] < 0 else "🟡"
    print(f"\n{impact_emoji} Impact Summary:")
    print(f"Future Canopy Cover: {result['baseline_prediction']:.1f}% → {result['scenario_prediction']:.1f}%")
    print(f"Net Change: {result['impact']:+.1f}% ({result['impact_percentage']:+.1f}%)")
    
    if abs(result['impact']) > 5:
        print("🚨 Significant impact detected!")
    elif abs(result['impact']) > 2:
        print("⚠️ Moderate impact detected.")
    else:
        print("ℹ️ Minor impact detected.")

# Create interactive sliders
print("🎛️ Interactive Scenario Explorer")
print("Use the sliders below to explore different what-if scenarios:")

interact(interactive_scenario_explorer,
         canopy_change=FloatSlider(min=-30, max=30, step=1, value=0, 
                                  description='Canopy Change (%):'),
         air_quality_change=FloatSlider(min=-50, max=50, step=5, value=0,
                                       description='Air Quality Change:'),
         soil_moisture_change=FloatSlider(min=-20, max=20, step=1, value=0,
                                         description='Soil Moisture Change (%):'),
         temperature_change=FloatSlider(min=-5, max=5, step=0.5, value=0,
                                       description='Temperature Change (°C):'))

## 10. Custom Scenario Testing
Define your own scenarios and test them

In [ ]:
def test_custom_scenario(scenario_name, **changes):
    """
    Test a custom scenario with specific changes
    
    Usage examples:
    test_custom_scenario("My Scenario", canopy_change=10, temperature_change=-1)
    test_custom_scenario("Drought Impact", soil_moisture_change=-15, temperature_change=2)
    """
    result = simulate_scenario(baseline_data, **changes)
    print_scenario_results(result, scenario_name)
    return result

# Example custom scenarios you can try:
print("🧪 Custom Scenario Examples:")
print("Run these scenarios by copying and modifying the code below:\n")

# Drought scenario
drought_scenario = test_custom_scenario(
    "Severe Drought",
    soil_moisture_change=-25,
    temperature_change=4,
    air_quality_change=15
)

# Air pollution reduction scenario
air_improvement_scenario = test_custom_scenario(
    "Air Quality Improvement Program",
    air_quality_change=-40,  # Significant air quality improvement
    canopy_change=5  # Slight increase in trees
)

# Perfect conditions scenario
perfect_conditions_scenario = test_custom_scenario(
    "Ideal Environmental Conditions",
    canopy_change=20,
    air_quality_change=-35,
    soil_moisture_change=15,
    temperature_change=-2
)

## 11. Key Insights and Recommendations
Summary of findings and actionable recommendations

In [ ]:
# Analyze feature importance and generate insights
print("🧠 KEY INSIGHTS FROM THE ECOSYSTEM MODEL")
print("="*60)

# Feature importance insights
importance_ranking = importance_df.sort_values('importance', ascending=False)
most_important = importance_ranking.iloc[0]['feature']
least_important = importance_ranking.iloc[-1]['feature']

print(f"\n1. 🎯 MOST INFLUENTIAL FACTOR: {most_important.replace('_', ' ').title()}")
print(f"   • Accounts for {importance_ranking.iloc[0]['importance']*100:.1f}% of prediction power")
print(f"   • This is the primary lever for ecosystem management")

print(f"\n2. 📊 FACTOR HIERARCHY:")
for i, row in importance_ranking.iterrows():
    stars = "⭐" * max(1, int(row['importance'] * 10))
    print(f"   • {row['feature'].replace('_', ' ').title()}: {stars} ({row['importance']*100:.1f}%)")

# Correlation insights
print(f"\n3. 🔗 STRONGEST CORRELATIONS WITH FUTURE CANOPY:")
strong_correlations = correlation_matrix['future_canopy_cover'].drop('future_canopy_cover').abs().sort_values(ascending=False)
for feature, corr in strong_correlations.head(3).items():
    direction = "positively" if correlation_matrix['future_canopy_cover'][feature] > 0 else "negatively"
    print(f"   • {feature.replace('_', ' ').title()}: {direction} correlated (r = {correlation_matrix['future_canopy_cover'][feature]:.3f})")

# Model performance insights
print(f"\n4. 🎯 MODEL PERFORMANCE:")
print(f"   • Can predict future conditions with {test_r2*100:.1f}% accuracy")
print(f"   • Average prediction error: ±{test_rmse:.1f}% canopy cover")
print(f"   • Model is {'reliable' if test_r2 > 0.8 else 'moderately reliable' if test_r2 > 0.6 else 'needs improvement'}")

print("\n\n💡 ACTIONABLE RECOMMENDATIONS")
print("="*60)

print("\n🌱 FOR URBAN PLANNERS:")
print("   1. Focus on increasing canopy cover - it has the highest impact")
print("   2. Monitor air quality closely as it significantly affects ecosystem health")
print("   3. Implement soil moisture management systems")
print("   4. Consider micro-climate effects when planning green spaces")

print("\n🏛️ FOR POLICY MAKERS:")
print("   1. Prioritize tree planting initiatives - our model shows significant positive impact")
print("   2. Regulate development to minimize canopy loss")
print("   3. Invest in air quality improvement programs")
print("   4. Create incentives for maintaining green cover")

print("\n👥 FOR COMMUNITY GROUPS:")
print("   1. Organize tree planting drives in your neighborhood")
print("   2. Advocate for better air quality measures")
print("   3. Support local environmental monitoring initiatives")
print("   4. Educate others about the interconnected nature of urban ecosystems")

# Calculate potential impact of different interventions
print("\n\n📈 INTERVENTION IMPACT ANALYSIS")
print("="*60)

interventions = {
    "Plant 100 new trees (15% canopy increase)": simulate_scenario(baseline_data, canopy_change=15),
    "Reduce air pollution by 30%": simulate_scenario(baseline_data, air_quality_change=-30),
    "Improve soil irrigation (10% moisture increase)": simulate_scenario(baseline_data, soil_moisture_change=10),
    "Combined green initiative": simulate_scenario(baseline_data, canopy_change=20, air_quality_change=-25, soil_moisture_change=12)
}

print("\nPotential impact of different interventions:")
for intervention, result in interventions.items():
    impact_emoji = "🟢" if result['impact'] > 0 else "🔴" if result['impact'] < 0 else "🟡"
    print(f"   {impact_emoji} {intervention}: {result['impact']:+.1f}% canopy improvement")

# Find the best intervention
best_intervention = max(interventions.items(), key=lambda x: x[1]['impact'])
print(f"\n🏆 MOST EFFECTIVE INTERVENTION: {best_intervention[0]}")
print(f"   Expected improvement: {best_intervention[1]['impact']:+.1f}% future canopy cover")

print("\n\n🔮 FUTURE WORK SUGGESTIONS")
print("="*60)
print("1. 📡 Collect real-time sensor data from actual urban parks")
print("2. 🌍 Expand model to include weather patterns and seasonal variations")
print("3. 🏘️ Scale model to neighborhood or city-wide predictions")
print("4. 🤖 Implement automated monitoring and alert systems")
print("5. 📱 Create a mobile app for community-based data collection")
print("6. 🎯 Add more ecosystem indicators (biodiversity, carbon sequestration)")

print("\n✅ Analysis complete! Use the interactive tools above to explore more scenarios.")

## 12. Conclusion and Next Steps

### 🎉 What We've Accomplished

1. **Built a Predictive Model**: Created a Random Forest model that can predict future canopy cover based on current environmental conditions

2. **Enabled What-If Analysis**: Developed tools to simulate different scenarios and their impacts on urban ecosystems

3. **Identified Key Factors**: Determined which environmental factors most influence ecosystem health

4. **Created Interactive Tools**: Built sliders and functions for easy exploration of different scenarios

5. **Generated Actionable Insights**: Provided specific recommendations for urban planners, policy makers, and communities

### 🚀 How to Use This Model

1. **For Quick Analysis**: Use the pre-defined scenarios in Section 8
2. **For Interactive Exploration**: Use the sliders in Section 9
3. **For Custom Scenarios**: Modify the code in Section 10
4. **For Decision Making**: Refer to the insights and recommendations in Section 11

### 📝 Model Limitations

- Based on simulated data (real sensor data would improve accuracy)
- Focuses on short-term predictions (seasonal and long-term climate effects not fully captured)
- Limited to specific environmental factors (could include more variables like soil pH, rainfall, etc.)

### 🔧 Customization Tips

- Replace the synthetic dataset with real data from your local park
- Adjust the model parameters in Section 5 for different accuracy vs. speed trade-offs
- Add more environmental factors by expanding the feature set
- Modify the scenario functions to test specific interventions relevant to your area

---

**Happy modeling! 🌳🌱** Feel free to experiment with different scenarios and share your findings with local environmental groups or city planners.